# SAADE AUA — Visualization & Audio Viewer (без повторного навчання)

Цей ноутбук **не навчає** моделі. Він читає `03_locked_splits.csv` і `04_val_predictions_ALL.csv` із вже виконаного запуску, показує всі сценарні графіки, аудіоплеєри та порівняння E1/E2/E3. Працює також зі старими 13-сценарними експериментами v1. Hidden test не відкривається автоматично.

In [ ]:
from pathlib import Path
import json, sys, pandas as pd
from IPython.display import display
WORK_DIR=Path.cwd()
sys.path.insert(0,str(WORK_DIR))
import saade_core as S
import saade_visuals as V

# ЗМІНІТЬ лише цей шлях на вже виконаний експеримент
RESULTS_DIR=Path('/home/jovyan/work/results/SAADE_AUA_2026_v2_5level')
assert (RESULTS_DIR/'03_locked_splits.csv').is_file(), 'Не знайдено 03_locked_splits.csv; перевірте RESULTS_DIR'
protocol_path=RESULTS_DIR/'06_FINAL_PROTOCOL.json'
if protocol_path.exists():
    protocol=json.loads(protocol_path.read_text(encoding='utf-8'))
    saved_cfg=protocol.get('config',{})
    profile=saved_cfg.get('scenario_profile','v1_3level')
    dataset_root=saved_cfg.get('root','/home/jovyan/work/AUA')
    seconds=float(saved_cfg.get('seconds',4))
    sr=int(saved_cfg.get('sample_rate',16000))
else:
    # Діагностика за кількістю різних сценаріїв у prediction CSV
    preview=pd.read_csv(RESULTS_DIR/'04_val_predictions_ALL.csv',usecols=['category','level'])
    profile='v2_5level' if int(preview.level.max())==5 else 'v1_3level'
    dataset_root='/home/jovyan/work/AUA';seconds=4;sr=16000
cfg=S.Config(root=dataset_root,out=str(RESULTS_DIR),scenario_profile=profile,seconds=seconds,sample_rate=sr)
splits=pd.read_csv(RESULTS_DIR/'03_locked_splits.csv')
pred=pd.read_csv(RESULTS_DIR/'04_val_predictions_ALL.csv')
print('Каталог:',RESULTS_DIR,'| Профіль:',profile,'| Записи:',len(splits),'| Прогнози:',len(pred))

## 1. Усі сценарії та параметри генерації

In [ ]:
guide=V.draw_scenario_guide(cfg,tag="viewer_validation")

## 2. Відтворення всіх сценаріїв для REAL / FAKE

In [ ]:
examples=V.pick_examples(splits,split='validation')
for row in examples:
    print('\n---', 'FAKE' if int(row.label)==1 else 'REAL','---')
    V.show_audio_gallery(row,cfg,show_players=True,show_figures=True,tag='viewer_validation')

## 3. Будь-який аудіозапис: інтерактивний плеєр

In [ ]:
player=V.make_interactive_player(splits,cfg,split="validation")

## 4. ПОВНИЙ сценарний аудит E1 / E2 / E3

In [ ]:
metrics,asri,coverage=V.report_scenario_tests(pred,cfg,tag='viewer_validation',show=True)
V.plot_roc_for_selected(pred,cfg,tag='viewer_validation')

## 5. Класифікація одного голосу під усіма впливами

In [ ]:
for row in examples:
    _=V.show_one_record_decisions(pred,row,cfg,tag='viewer_validation')

## 6. Незалежний hidden test — лише за явного рішення дослідника

Не використовуйте цей блок для вибору налаштувань. Hidden test можна переглянути **тільки після завершення експерименту та фіксації протоколу**. Змінна `ALLOW_HIDDEN_VIEW=False` за замовчуванням.

In [ ]:
ALLOW_HIDDEN_VIEW=False
if ALLOW_HIDDEN_VIEW:
    hp=RESULTS_DIR/'07_hidden_predictions_ALL.csv'
    if not hp.exists():
        print('Hidden predictions відсутні — фінальний тест ще не виконаний.')
    else:
        hidden=pd.read_csv(hp)
        hm,ha,hc=V.report_scenario_tests(hidden,cfg,tag='viewer_hidden_test',show=True)
else:
    print('Hidden test не переглядається.')

## 7. Пакунок графіків і таблиць (без аудіо)

In [ ]:
archive=S.zip_results(cfg,include_models=False)
print('ZIP зі звітами:',archive)